# Modelo de Clasificación — Iteración 3: código generado por máquina

Adaptación del modelo de la iteración 2 al **nuevo dataset en parquet**
(`new_data/`): la tarea ya no es comparar *pares* de archivos sino
clasificar cada snippet individual — **0 = humano, 1 = generado por
máquina (LLM)**.

- **La arquitectura del modelo NO cambia** (misma red del paper basado en
  Dolos: Dense 32 → Dropout 0.2 → Dense 16 → sigmoide, Adam 1e-3,
  binary cross-entropy).
- **Lo que cambia es cómo se alimentan los datos**: en lugar de las 9
  características de similitud por par, se calculan 16 características
  *intrínsecas* por snippet con la misma maquinaria
  (tokenización AST normalizada + Winnowing) más métricas de estilo
  (`pipeline/snippet_features.py`).
- Se usan los **splits oficiales** del dataset: entrenamiento (500k),
  validación (100k, para early stopping y umbral) y muestra de prueba
  (1k, nunca vista).

Antes de ejecutar este notebook hay que generar las características:

```bash
python -m pipeline.build_snippets --input new_data/task_a_training_set_1.parquet \
    --output new_data_features/train_features.parquet --k 15 --w 4 --workers 10
python -m pipeline.build_snippets --input new_data/task_a_validation_set.parquet \
    --output new_data_features/validation_features.parquet --k 15 --w 4 --workers 10
python -m pipeline.build_snippets --input new_data/task_a_test_set_sample.parquet \
    --output new_data_features/test_features.parquet --k 15 --w 4 --workers 10
```

## 1. Importar librerías

In [ ]:
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
)

sys.path.insert(0, ".")   # para importar pipeline/ desde el notebook
from pipeline.snippet_features import SNIPPET_FEATURE_COLUMNS

## 2. Configuración

In [ ]:
RANDOM_SEED = 42
LABEL_COLUMN = "label"

# Parámetros del pipeline con los que se generaron las características
K, W, MASKING = 15, 4, "medium"

# Línea base de Dolos (F1 reportado en el paper de referencia)
DOLOS_F1 = 0.865

FEATURES_DIR = "new_data_features"
TRAIN_PATH = os.path.join(FEATURES_DIR, "train_features.parquet")
VAL_PATH = os.path.join(FEATURES_DIR, "validation_features.parquet")
TEST_PATH = os.path.join(FEATURES_DIR, "test_features.parquet")

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"{len(SNIPPET_FEATURE_COLUMNS)} características: {SNIPPET_FEATURE_COLUMNS}")

## 3. Cargar los datos

El dataset trae sus propios splits, así que **no** se hace
`train_test_split`: se entrena con el de entrenamiento, la validación
oficial controla el early stopping y el umbral, y la muestra de prueba
solo se toca al final.

In [ ]:
def load_features(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No existe {path}. Genera las características con "
            "`python -m pipeline.build_snippets` (ver celda de arriba)."
        )
    df = pd.read_parquet(path)
    x = df[SNIPPET_FEATURE_COLUMNS].values.astype("float32")
    y = df[LABEL_COLUMN].values.astype("float32")
    return x, y, df


x_train, y_train, df_train = load_features(TRAIN_PATH)
x_val, y_val, df_val = load_features(VAL_PATH)
x_test, y_test, df_test = load_features(TEST_PATH)

for name, xs, ys in [("train", x_train, y_train),
                     ("validation", x_val, y_val),
                     ("test", x_test, y_test)]:
    print(f"{name:10s}: {xs.shape[0]:7d} snippets  "
          f"(máquina: {int(ys.sum()):6d}, humano: {int((1 - ys).sum()):6d})")

## 4. Construir el modelo

**Idéntico** al de la iteración 2 — solo cambia `input_dim`
(16 características por snippet en lugar de 9 por par).

In [ ]:
def build_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc"),
        ],
    )
    return model


def make_callbacks():
    """Early stopping: corta el entrenamiento cuando val_loss deja de
    mejorar y restaura los mejores pesos."""
    return [keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True,
    )]


def f1_from(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


build_model(len(SNIPPET_FEATURE_COLUMNS)).summary()

## 5. Entrenamiento

Con 500k ejemplos el `batch_size` sube de 32 a 512 (cambio en la
alimentación de datos, no en la arquitectura); el early stopping
monitorea la **validación oficial** del dataset.

In [ ]:
scaler = StandardScaler()
x_train_s = scaler.fit_transform(x_train)
x_val_s = scaler.transform(x_val)
x_test_s = scaler.transform(x_test)

model = build_model(x_train.shape[1])
history = model.fit(
    x_train_s, y_train,
    validation_data=(x_val_s, y_val),
    epochs=50,
    batch_size=512,
    callbacks=make_callbacks(),
    verbose=2,
)
print(f"Épocas entrenadas: {len(history.history['loss'])}")

## 6. Umbral de decisión sobre validación (sección 8.2 del marco)

Mismo protocolo de la iteración 2: barrer umbrales y elegir el que
maximiza F1 en validación (ante empate, el más cercano a 0.5).

In [ ]:
val_probs = model.predict(x_val_s, verbose=0).ravel()
thresholds = np.linspace(0.05, 0.95, 91)
f1_by_threshold = np.array([f1_score(y_val, val_probs >= t) for t in thresholds])
candidates = thresholds[f1_by_threshold >= f1_by_threshold.max() - 1e-9]
best_threshold = float(candidates[np.argmin(np.abs(candidates - 0.5))])
print(f"Umbral óptimo (validación): {best_threshold:.2f} "
      f"(F1 val = {f1_by_threshold.max():.3f})")

## 7. Métricas: Precision · Recall · F1 · AUC-ROC

Se reportan sobre la **validación** (misma distribución que el
entrenamiento) y sobre la **muestra de prueba** (1 000 snippets con
otra mezcla de lenguajes y generadores — mide la generalización).

In [ ]:
def evaluate_split(name, y_true, probs, threshold):
    y_pred = (probs >= threshold).astype(int)
    return {
        "split": name,
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "auc_roc": roc_auc_score(y_true, probs),
    }


test_probs = model.predict(x_test_s, verbose=0).ravel()

val_metrics = evaluate_split("validación", y_val, val_probs, best_threshold)
test_metrics = evaluate_split("prueba", y_test, test_probs, best_threshold)

results = pd.DataFrame([val_metrics, test_metrics]).set_index("split").round(3)
print(results)

print(f"\nLínea base Dolos: F1 = {DOLOS_F1} → "
      f"validación {'supera' if val_metrics['f1'] >= DOLOS_F1 else 'NO supera'} | "
      f"prueba {'supera' if test_metrics['f1'] >= DOLOS_F1 else 'NO supera'}")

## 7b. Desglose por lenguaje (muestra de prueba)

In [ ]:
y_pred_test = (test_probs >= best_threshold).astype(int)
by_lang = df_test.assign(pred=y_pred_test).groupby("language").apply(
    lambda g: pd.Series({
        "n": len(g),
        "positivos": int(g["label"].sum()),
        "f1": f1_score(g["label"], g["pred"]) if g["label"].sum() else np.nan,
        "accuracy": (g["label"] == g["pred"]).mean(),
    }),
    include_groups=False,
)
print(by_lang.round(3))

## 8. Matriz de confusión

In [ ]:
y_pred_val = (val_probs >= best_threshold).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, y_true, y_pred) in zip(axes, [
    ("Validación (100k)", y_val, y_pred_val),
    ("Prueba (1k, nunca vista)", y_test, y_pred_test),
]):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=["Humano", "Máquina"]
    )
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"Matriz de confusión — {name}")
plt.tight_layout()
os.makedirs("results", exist_ok=True)
plt.savefig("results/confusion_matrix_new_data.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. F1 vs. línea base Dolos (0.865)

Izquierda: las 4 métricas clave por split. Derecha: gráfica de barras
del F1 con la puntuación de **Dolos (0.865)** como barra y línea de
referencia.

In [ ]:
metric_names = ["precision", "recall", "f1", "auc_roc"]
xpos = np.arange(len(metric_names))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ── Panel 1: todas las métricas, validación vs prueba ──
bars_val = ax1.bar(xpos - width / 2, [val_metrics[m] for m in metric_names],
                   width, label="Validación", color="#1f77b4")
bars_test = ax1.bar(xpos + width / 2, [test_metrics[m] for m in metric_names],
                    width, label="Prueba", color="#ff7f0e")
ax1.hlines(DOLOS_F1, xmin=1.55, xmax=2.45, colors="crimson",
           linestyles="--", linewidth=1.8, label=f"Dolos F1 = {DOLOS_F1}")
ax1.set_xticks(xpos)
ax1.set_xticklabels([m.upper().replace("_", "-") for m in metric_names])
ax1.set_ylim(0, 1.05)
ax1.set_ylabel("Valor")
ax1.set_title("Métricas del modelo por split")
ax1.grid(axis="y", linestyle="--", alpha=0.3)
ax1.legend()
for bars in [bars_val, bars_test]:
    for bar in bars:
        h = bar.get_height()
        ax1.annotate(f"{h:.3f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                     xytext=(0, 3), textcoords="offset points",
                     ha="center", va="bottom", fontsize=8)

# ── Panel 2: F1 del modelo vs Dolos ──
f1_labels = ["Modelo\n(validación)", "Modelo\n(prueba)", "Dolos\n(baseline)"]
f1_values = [val_metrics["f1"], test_metrics["f1"], DOLOS_F1]
colors = ["#1f77b4", "#ff7f0e", "crimson"]
bars_f1 = ax2.bar(f1_labels, f1_values, color=colors, width=0.55)
ax2.axhline(DOLOS_F1, color="crimson", linestyle="--", linewidth=1.5, alpha=0.7)
ax2.set_ylim(0, 1.05)
ax2.set_ylabel("F1-score")
ax2.set_title(f"F1-score vs. línea base Dolos ({DOLOS_F1})")
ax2.grid(axis="y", linestyle="--", alpha=0.3)
for bar in bars_f1:
    h = bar.get_height()
    ax2.annotate(f"{h:.3f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                 xytext=(0, 3), textcoords="offset points",
                 ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("results/f1_vs_dolos_new_data.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Guardar artefactos

In [ ]:
import joblib

model.save("ai_code_model.keras")
joblib.dump(scaler, "scaler_new_data.joblib")
with open("decision_threshold_new_data.json", "w") as f:
    json.dump({
        "threshold": best_threshold,
        "k": K, "w": W, "masking": MASKING,
        "feature_columns": SNIPPET_FEATURE_COLUMNS,
        "dolos_f1_baseline": DOLOS_F1,
        "validation_metrics": {k_: round(v, 4) for k_, v in val_metrics.items() if k_ != "split"},
        "test_metrics": {k_: round(v, 4) for k_, v in test_metrics.items() if k_ != "split"},
    }, f, indent=2)

print("Guardado: ai_code_model.keras, scaler_new_data.joblib, "
      "decision_threshold_new_data.json")